In [2]:
import torch
inputs = torch.tensor(
    [[0.43, 0.15, 0.89],
     [0.55, 0.87, 0.66],
     [0.57, 0.85, 0.64],
     [0.22, 0.58, 0.33],
     [0.77, 0.25, 0.10],
     [0.05, 0.80, 0.55]]
)
query = inputs[1]
attn_score_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_score_2[i] = torch.dot(query, x_i)
attn_weights_2_tmp = attn_score_2 / attn_score_2.sum()

def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum()

print(torch.softmax(attn_score_2, dim=0))

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "c:\ProgramData\anaconda3\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "c:\ProgramData\anaconda3\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "c:\ProgramData\anaconda3\lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\ProgramData\anaconda3\lib\site-packages\traitlets\config\application.py", line 992, in launch_instance
    app.start()
  File "c:\ProgramData\anaconda3\lib\site-pack

1.计算注意力分数 <br>
2.计算注意力权重 <br>
3.计算上下文向量 <br>

In [ ]:
attn_score = torch.empty(inputs.shape[0], inputs.shape[0])
for i, query in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_score[i][j] += torch.dot(query, x_j)
print(attn_score)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [ ]:
attn_score = inputs @ inputs.T
attn_weights = torch.softmax(attn_score, dim=-1)
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [25]:
context_vec = attn_weights @ inputs
print(context_vec)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


## 可训练自注意力机制

In [28]:
x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2

In [31]:
torch.manual_seed(123)
w_q = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
w_k = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
w_v = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
query_2 = x_2 @ w_q
key_2 = x_2 @ w_k
value_2 = x_2 @ w_v
print(query_2)

tensor([0.4306, 1.4551])


In [ ]:
query = inputs @ w_q
key = inputs @ w_k
value = inputs @ w_v


In [41]:
attn_score_2 = query[1] @ key.T
print(attn_score_2)
d_k = key.shape[-1]
attn_weight_2 = torch.softmax(attn_score_2 / d_k ** 0.5, dim=-1)
print(attn_weight_2)
context_vec_2 = attn_weight_2 @ value
print(context_vec_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])
tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])
tensor([0.3061, 0.8210])


In [1]:
import torch.nn as nn
class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.w_q = nn.Parameter(torch.rand(d_in, d_out), requires_grad=True)
        self.w_k = nn.Parameter(torch.rand(d_in, d_out), requires_grad=True)
        self.w_v = nn.Parameter(torch.rand(d_in, d_out), requires_grad=True)

    def forward(self, x):
        query = x @ self.w_q
        key = x @ self.w_k
        value = x @ self.w_v

        attn_score = query @ key.T
        d_k = key.shape[-1]
        attn_weights = torch.softmax(attn_score / d_k ** 0.5, dim=-1)
        context_vec = attn_weights @ value
        return context_vec, attn_weights

In [20]:
torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in=3, d_out=2)
context_vec_v1, attn_weights_v1 = sa_v1(inputs)
print(context_vec_v1)

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


In [10]:
import torch.nn as nn
class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, bias=False):
        super().__init__()
        self.w_q = nn.Linear(d_in, d_out, bias=bias)
        self.w_k = nn.Linear(d_in, d_out, bias=bias)
        self.w_v = nn.Linear(d_in, d_out, bias=bias)

    def forward(self, x):
        query = self.w_q(x)
        key = self.w_k(x)
        value = self.w_v(x)

        attn_score = query @ key.T
        d_k = key.shape[-1]
        attn_weights = torch.softmax(attn_score / d_k ** 0.5, dim=-1)
        context_vec = attn_weights @ value
        return context_vec, attn_weights

torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in=3, d_out=2)
context_vec_v2, attn_weights_v2 = sa_v2(inputs)
print(context_vec_v2)

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


In [32]:
sa_v1.w_q = nn.Parameter(sa_v2.w_q.weight.T)
sa_v1.w_k = nn.Parameter(sa_v2.w_k.weight.T)
sa_v1.w_v = nn.Parameter(sa_v2.w_v.weight.T)
context_vec_v1, attn_weights_v1 = sa_v1(inputs)
print(context_vec_v1)

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


## 因果注意力

In [48]:
queries = sa_v2.w_q(inputs)
keys = sa_v2.w_k(inputs)
values = sa_v2.w_v(inputs)

attn_score = queries @ keys.transpose(-2, -1)
attn_weights = torch.softmax(attn_score / keys.shape[-1] ** 0.5, dim=-1)
print(attn_weights)

tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


In [37]:
context_length = attn_weights.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
mask_weight = attn_weights * mask_simple
row_sums = mask_weight.sum(dim=-1, keepdim=True)
mask_weight_norm = mask_weight / row_sums
print(mask_weight_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<DivBackward0>)


In [41]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_score.masked_fill(mask == 1, float('-inf'))
attn_weights_norm = torch.softmax(masked / keys.shape[-1] ** 0.5, dim=-1)
print(attn_weights_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


In [44]:
torch.manual_seed(123)
dropout = torch.nn.Dropout(p=0.5)
example  = torch.ones(6, 6)
print(dropout(attn_weights_norm))

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.7599, 0.6194, 0.6206, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4921, 0.4925, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3966, 0.0000, 0.3775, 0.0000, 0.0000],
        [0.0000, 0.3327, 0.3331, 0.3084, 0.3331, 0.0000]],
       grad_fn=<MulBackward0>)


In [ ]:
batch = torch.stack((inputs, inputs), dim=0)


torch.Size([2, 6, 3])


In [59]:
class CausalAttention(torch.nn.Module):
    def __init__(self, input_dim, output_dim, context_length, dropout, bias=False):
        super().__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim

        self.w_q = nn.Linear(input_dim, output_dim, bias=bias)
        self.w_k = nn.Linear(input_dim, output_dim, bias=bias)
        self.w_v = nn.Linear(input_dim, output_dim, bias=bias)

        self.dropout = nn.Dropout(p=dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )
    
    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.w_k(x)
        queries = self.w_q(x)
        values = self.w_v(x)
        attn_score = queries @ keys.transpose(1, 2)
        attn_score.masked_fill_(self.mask == 1, -torch.inf)
        attn_weight = torch.softmax(attn_score / keys.shape[-1] ** 0.5, dim=-1)
        attn_weight = self.dropout(attn_weight)

        context = attn_weight @ values
        return context


In [52]:
torch.manual_seed(123)
context_length = batch.shape[1]
causal_attn = CausalAttention(
    input_dim=3,
    output_dim=2,
    context_length=context_length,
    dropout=0.5,
    bias=False
)
context_vec = causal_attn(batch)
print(context_vec.shape)

torch.Size([2, 6, 2])
torch.Size([2, 2, 6])
torch.Size([2, 6, 2])


## 多头注意力机制

In [75]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [
                CausalAttention(
                    input_dim=d_in,
                    output_dim=d_out,
                    context_length=context_length,
                    dropout=dropout,
                    bias=bias
                )
                for _ in range(num_heads)
            ]
        )

    def forward(self, x):
        return torch.cat([h(x) for h in self.heads], dim=-1)


In [77]:
torch.manual_seed(123)
context_length = batch.shape[1]
multi_attn = MultiHeadAttentionWrapper(
    d_in=3,
    d_out=1,
    context_length=context_length,
    dropout=0.5,
    num_heads=2,
    bias=False
)
context_vec = multi_attn(batch)
print(context_vec.shape)

torch.Size([2, 6, 2])


In [90]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        self.input_dim = d_in
        self.output_dim = d_out
        self.nums_heads = num_heads
        self.head_dim = d_out // self.nums_heads

        self.w_q = nn.Linear(self.input_dim, self.output_dim, bias=bias)
        self.w_k = nn.Linear(self.input_dim, self.output_dim, bias=bias)
        self.w_v = nn.Linear(self.input_dim, self.output_dim, bias=bias)

        self.out_proj = nn.Linear(self.output_dim, self.output_dim, bias=bias)
        self.dropout = nn.Dropout(p=dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.w_k(x)
        queries = self.w_q(x)
        values = self.w_v(x)
        keys = keys.view(b, num_tokens, self.nums_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.nums_heads, self.head_dim)
        values = values.view(b, num_tokens, self.nums_heads, self.head_dim)

        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)
        attn_score = queries @ keys.transpose(2, 3)
        attn_score = attn_score.masked_fill(self.mask[0:num_tokens, 0:num_tokens] == 0, float("-inf"))
        attn_weight = torch.softmax(attn_score / keys.shape[-1] ** 0.5, dim=-1)
        attn_weight = self.dropout(attn_weight)

        context_vec = (attn_weight @ values).transpose(1, 2)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.output_dim)
        context_vec = self.out_proj(context_vec)
        return context_vec


In [89]:
torch.manual_seed(123)
d_in = 768
d_out = 768
context_length = 1024
batch = torch.stack(
    [torch.rand(d_in, d_out) for _ in range(4)],
    dim=0
)
print(batch.shape)
multi_attn = MultiHeadAttention(
    d_in=d_in,
    d_out=d_out,
    context_length=context_length,
    dropout=0.5,
    num_heads=12,
    bias=False
)
context_vec = multi_attn(batch)
print(context_vec.shape)

torch.Size([4, 768, 768])


RuntimeError: The size of tensor a (1024) must match the size of tensor b (768) at non-singleton dimension 3